# Document OCR with `indic-ocr`

Extract text and layout from a scanned page in any Indian script. Returns Markdown plus a list of layout blocks with bounding boxes and confidence.

| | |
|---|---|
| Endpoint | `POST /v1/chat/completions` (OpenAI-compatible, image input) |
| Model | `indic-ocr` |
| Input | one PNG or JPEG page as a base64 data URL; optional `table_format` (`html` default, `markdown`), `max_tokens` |
| Output | `choices[0].message.content` (Markdown) + `blocks[]` (order, label, type, bbox_xyxy, conf, text) |
| Billing | per page |

In [ ]:
%pip install -q requests==2.32.3 pillow==10.4.0

In [ ]:
import os, json, requests

BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
API_KEY = os.environ["BODHAN_API_KEY"]  # export BODHAN_API_KEY=... before starting Jupyter
HEADERS = {"Authorization": f"Bearer {API_KEY}"}


def raise_for_bodhan(resp):
    """Bodhan errors are JSON: {"error": {"message", "code", "request_id"}}. Surface them readably."""
    if resp.ok:
        return resp
    try:
        err = resp.json()["error"]
        raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')} (request_id={err.get('request_id')})")
    except (ValueError, KeyError):
        resp.raise_for_status()

## 1. Make a test page

No scanner handy? Render one with Pillow. Replace `PAGE` with your own PNG/JPEG to OCR a real document.

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
PAGE = OUT / "test_page.png"

img = Image.new("RGB", (900, 400), "white")
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("/System/Library/Fonts/Supplemental/Kohinoor Devanagari.ttc", 36)  # macOS; adjust on Linux
except OSError:
    font = ImageFont.load_default()
draw.text((40, 40), "विद्यालय सूचना", font=font, fill="black")
draw.text((40, 120), "कल विद्यालय बंद रहेगा।", font=font, fill="black")
draw.text((40, 200), "Contact: 011-2345-6789", font=font, fill="black")
img.save(PAGE)
img

## 2. OCR the page

In [ ]:
import base64, mimetypes


def ocr_page(path: str | Path, table_format: str = "markdown", max_tokens: int | None = None) -> dict:
    mime = mimetypes.guess_type(str(path))[0] or "image/png"
    b64 = base64.b64encode(Path(path).read_bytes()).decode()
    body = {
        "model": "indic-ocr",
        "messages": [{"role": "user", "content": [{"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}}]}],
        "table_format": table_format,
    }
    if max_tokens:
        body["max_tokens"] = max_tokens
    resp = requests.post(f"{BASE_URL}/v1/chat/completions", headers={**HEADERS, "Content-Type": "application/json"}, json=body, timeout=180)
    return raise_for_bodhan(resp).json()


result = ocr_page(PAGE)
print(result["choices"][0]["message"]["content"])

## 3. Layout blocks

Every block carries its reading `order`, a layout `label` (title, paragraph, table, figure, ...), a pixel bounding box `[x1, y1, x2, y2]`, and a confidence.

In [ ]:
for b in result.get("blocks", []):
    print(f"{b['order']:2} {b['label']:14} conf={b['conf']:.2f} bbox={b['bbox_xyxy']}  {b['text'][:60]!r}")

Draw the boxes back onto the page to sanity-check the layout.

In [ ]:
annotated = Image.open(PAGE).convert("RGB")
d = ImageDraw.Draw(annotated)
for b in result.get("blocks", []):
    x1, y1, x2, y2 = b["bbox_xyxy"]
    d.rectangle([x1, y1, x2, y2], outline="red", width=2)
    d.text((x1, max(0, y1 - 14)), b["label"], fill="red")
annotated.save(OUT / "annotated.png")
annotated

## 4. Multi-page PDFs

The endpoint takes one page image per request. Rasterise PDFs first (`pdf2image` + poppler, or `PyMuPDF`), then call `ocr_page` per page. Page images around 150–200 DPI are a good trade-off between accuracy and upload size.

## 5. Tables

Set `table_format="html"` to get `<table>` markup that survives merged cells; `"markdown"` is easier to read but flattens complex tables.

**Next:** pass the Markdown to [`translate`](../translate/translate.ipynb) to read documents in another language.